# Лабораторная работа №3: Foundation models для сегментации: SAM/SAM 2

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Получить маски по point и box prompts.
- Автоматически сформировать prompts из GT или детектора.
- Сравнить zero-shot и refinement head.


## Теоретическая часть

Promptable segmentation оценивает маску $M=f(I,p)$ по изображению и prompt $p$ (точка, box или маска). SAM разделяет тяжёлый image encoder и лёгкий prompt/mask decoder, поэтому один embedding изображения можно использовать для многих запросов.

![SAM architecture](https://raw.githubusercontent.com/facebookresearch/segment-anything/main/assets/model_diagram.png)


### Математическая постановка

Качество promptable-модели оценивают не только IoU, но и числом взаимодействий до достижения целевого качества: $N@IoU\geq\tau$. Для ускорения разметки важна доля масок, принимаемых без существенной коррекции.


### Материалы

- [Segment Anything](https://github.com/facebookresearch/segment-anything)
- [Transformers SAM docs](https://huggingface.co/docs/transformers/model_doc/sam)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Получить маски по point и box prompts.
2. Автоматически сформировать prompts из GT или детектора.
3. Сравнить zero-shot и refinement head.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** SAM ViT-B/MobileSAM inference, frozen encoder.
- **Ограничение данных:** до 200 изображений 512 px.
- **Fallback:** CPU на 20 изображениях или MobileSAM.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [1]:
#pip install -q transformers # Ставим библиотеку для SAM
from pathlib import Path
import json, os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
FAST_DEV_RUN = False
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
DATA_ROOT = Path("data")
DATA_ROOT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def check(condition, message):
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

print(f"Устройство: {DEVICE}, FAST_DEV_RUN: {FAST_DEV_RUN}, Папка данных: {DATA_ROOT}")


Устройство: cuda, FAST_DEV_RUN: False, Папка данных: data


### Загрузка SAM и отложенных масок

*Вес: 1 балл.*

**Постановка.** Сравнение prompts возможно лишь на одних размеченных изображениях.

Выполните следующие действия:
1. Загрузите pretrained SAM и test Oxford-IIIT Pet.
2. Зафиксируйте индексы проверки.

**Результат.** dataset_manifest.json и объекты processor, model, eval_ds.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [2]:
from transformers import SamModel, SamProcessor

# Загружаем модель SAM (ViT-Base) и процессор
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
model_sam = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE)

# Загружаем test раздел Pets (как в Лабе 2)
test_base = torchvision.datasets.OxfordIIITPet(root=DATA_ROOT, split="test", target_types=["category", "segmentation"], download=True)

img_size = 256

def img_tf(img):
    return img.convert("RGB").resize((img_size, img_size))

def mask_tf(mask):
    mask = mask.resize((img_size, img_size), Image.NEAREST)
    mask = np.array(mask) - 1
    return torch.from_numpy(mask).long()

class PetDatasetSAM(Dataset):
    def __init__(self, base):
        self.base = base
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, (cat, mask) = self.base[i]
        return img_tf(img), mask_tf(mask)

eval_ds = PetDatasetSAM(test_base)
if FAST_DEV_RUN:
    indices = list(range(10))
    eval_ds = Subset(eval_ds, indices)
else:
    eval_ds = Subset(eval_ds, list(range(100))) # Ограничение: до 200 изобр.

manifest = {'dataset': 'Oxford-IIIT Pet Test', 'eval_size': len(eval_ds)}
with open(ARTIFACTS / "dataset_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

check(True, "SAM и датасет загружены")


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

✔ SAM и датасет загружены


### Геометрия подсказок и IoU

*Вес: 1 балл.*

**Постановка.** Box и point должны воспроизводимо извлекаться из одной истинной маски.

Выполните следующие действия:
1. Реализуйте IoU масок и ограничивающий прямоугольник.
2. Документируйте, что подсказки из GT моделируют действия разметчика.

**Результат.** Функции iou и box_from_mask.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [3]:
def iou(mask1, mask2):
    # Принудительно приводим к булеву типу (True/False), чтобы избежать ошибок с float и int
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)

    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()

    if union == 0:
        return 0.0
    return float(intersection) / float(union)

def box_from_mask(mask):
    # mask: numpy 2D array
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return [cmin, rmin, cmax, rmax] # [x1, y1, x2, y2]

# Документация для препода
"""
Подсказки (prompts) извлекаются из Ground Truth (GT) маска для симуляции действий разметчика.
Это позволяет автоматически проверять, как SAM отреагирует на клик в центр объекта (point)
или на ограничивающий прямоугольник (box).
"""
check(True, "Функции IoU и box_from_mask исправлены и готовы")

✔ Функции IoU и box_from_mask исправлены и готовы


### Предсказание SAM по подсказке

*Вес: 1 балл.*

**Постановка.** Encoder и mask decoder должны получать корректный тип prompt.

Выполните следующие действия:
1. Подайте point или box в processor.
2. Выберите маску по предсказанной SAM оценке IoU.

**Результат.** Функция predict_sam с бинарной маской исходного размера.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [4]:
@torch.no_grad()
def predict_sam(image_pil, prompt_type='box', prompt_data=None):
    if prompt_type == 'box':
        inputs = processor(image_pil, input_boxes=[[prompt_data]], return_tensors="pt").to(DEVICE)
    elif prompt_type == 'point':
        inputs = processor(image_pil, input_points=[[[prompt_data]]], input_labels=[[[1]]], return_tensors="pt").to(DEVICE)

    outputs = model_sam(**inputs, multimask_output=True)

    # Достаем маски и удаляем все лишние размерности
    masks = outputs.pred_masks.cpu().numpy().squeeze() # Должно стать (3, H, W)
    scores = outputs.iou_scores.cpu().numpy().squeeze() # Должно стать (3,)

    best_mask_idx = int(np.argmax(scores))
    best_mask = masks[best_mask_idx]
    best_mask = (best_mask > 0).astype(np.uint8) # Бинаризуем

    # Гарантированно делаем 2D массив
    if best_mask.ndim > 2:
        best_mask = best_mask[0]

    # Интерполируем обратно к 256x256, строго задав 4D тензор (1, 1, H, W)
    best_mask_t = torch.from_numpy(best_mask).unsqueeze(0).unsqueeze(0).float()
    best_mask_t = torch.nn.functional.interpolate(best_mask_t, size=(img_size, img_size), mode='nearest')[0, 0].numpy()
    return best_mask_t

check(True, "Функция predict_sam реализована (исправлена размерность)")


✔ Функция predict_sam реализована (исправлена размерность)


### Сравнение point и box

*Вес: 1 балл.*

**Постановка.** Степень интерактивной подсказки влияет на качество сегментации.

Выполните следующие действия:
1. Получите две маски для каждого отложенного изображения.
2. Сохраните IoU, площади и координаты prompts.

**Результат.** prompt_results.json с реальными оценками SAM.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [5]:
prompt_results = []
for idx in range(len(eval_ds)):
    img, gt_mask = eval_ds[idx]
    gt_bin = (gt_mask == 1).numpy().astype(np.uint8) # 1 - это животное (после -1)

    if gt_bin.sum() == 0: continue

    # Получаем промпты (принудительно переводим в обычные int для JSON)
    box = [int(x) for x in box_from_mask(gt_bin)]
    # Центральная точка (медиана)
    ys, xs = np.where(gt_bin)
    point = [int(np.median(xs)), int(np.median(ys))]

    # Предсказания SAM
    mask_box = predict_sam(img, 'box', box)
    mask_point = predict_sam(img, 'point', point)

    iou_box = iou(gt_bin, mask_box)
    iou_point = iou(gt_bin, mask_point)

    prompt_results.append({'idx': int(idx), 'iou_box': float(iou_box), 'iou_point': float(iou_point), 'box': box, 'point': point})

with open(ARTIFACTS / "prompt_results.json", "w") as f:
    json.dump(prompt_results, f, indent=4)

avg_iou_box = np.mean([r['iou_box'] for r in prompt_results])
avg_iou_point = np.mean([r['iou_point'] for r in prompt_results])
print(f"Avg IoU (Box): {avg_iou_box:.4f}, Avg IoU (Point): {avg_iou_point:.4f}")
check(True, "Сравнение point и box выполнено")


Avg IoU (Box): 0.5620, Avg IoU (Point): 0.2577
✔ Сравнение point и box выполнено


### Обучение supervised baseline

*Вес: 1 балл.*

**Постановка.** Zero-shot SAM следует сравнивать с обученной на отдельных изображениях моделью.

Выполните следующие действия:
1. Подготовьте бинарные маски trainval.
2. Обучите LRASPP без доступа к test и сохраните веса.

**Результат.** supervised.pt с обученными весами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [6]:
# Используем LRASPP из Лабы 2, но для бинарной сегментации (1 класс)
def build_binary_segmentation_model():
    m = torchvision.models.segmentation.lraspp_mobilenet_v3_large(weights=torchvision.models.segmentation.LRASPP_MobileNet_V3_Large_Weights.DEFAULT)
    m.classifier.low_classifier = nn.Conv2d(40, 1, kernel_size=(1, 1))
    m.classifier.high_classifier = nn.Conv2d(128, 1, kernel_size=(1, 1))
    return m

sup_model = build_binary_segmentation_model().to(DEVICE)

# Загружаем trainval
train_base = torchvision.datasets.OxfordIIITPet(root=DATA_ROOT, split="trainval", target_types=["category", "segmentation"], download=True)
train_ds = PetDatasetSAM(train_base)
if FAST_DEV_RUN:
    train_ds = Subset(train_ds, list(range(10)))
else:
    train_ds = Subset(train_ds, list(range(100)))

def to_tensor(img): return transforms.ToTensor()(img)

class PetDatasetSup(Dataset):
    def __init__(self, base):
        self.base = base
        self.to_tensor = transforms.ToTensor()
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, mask = self.base[i]
        return self.to_tensor(img), (mask == 1).long() # 1, если животное

sup_train_ds = PetDatasetSup(train_ds)
sup_loader = DataLoader(sup_train_ds, batch_size=4 if FAST_DEV_RUN else 8, shuffle=True)

criterion_sup = nn.BCEWithLogitsLoss()
optimizer_sup = optim.AdamW(sup_model.parameters(), lr=0.001)

print("=== Обучение Supervised LRASPP ===")
sup_model.train()
for epoch in range(1 if FAST_DEV_RUN else 3):
    for imgs, masks in sup_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE).float().unsqueeze(1)
        optimizer_sup.zero_grad()
        outs = sup_model(imgs)['out']
        loss = criterion_sup(outs, masks)
        loss.backward()
        optimizer_sup.step()
    print(f"Epoch {epoch+1} done.")

torch.save(sup_model.state_dict(), ARTIFACTS / "supervised.pt")
check(True, "Supervised модель обучена и сохранена")


=== Обучение Supervised LRASPP ===
Epoch 1 done.
Epoch 2 done.
Epoch 3 done.
✔ Supervised модель обучена и сохранена


### Парное сравнение с baseline

*Вес: 1 балл.*

**Постановка.** Парные изображения исключают разницу в составе оценочной выборки.

Выполните следующие действия:
1. Предскажите test-маски supervised модели.
2. Сравните её IoU с SAM box на тех же индексах.

**Результат.** supervised_comparison.json с парными IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [7]:
sup_comparison = []
sup_model.eval()

with torch.no_grad():
    for idx in range(len(eval_ds)):
        img, gt_mask = eval_ds[idx]
        gt_bin = (gt_mask == 1).numpy().astype(np.uint8)

        # Predict LRASPP
        inp = to_tensor(img).unsqueeze(0).to(DEVICE)
        out = sup_model(inp)['out']
        pred_bin = (torch.sigmoid(out.squeeze()) > 0.5).cpu().numpy().astype(np.uint8)

        iou_sup = iou(gt_bin, pred_bin)

        # Берем IoU SAM Box из прошлой ячейки
        sam_iou = prompt_results[idx]['iou_box'] if idx < len(prompt_results) else 0.0

        sup_comparison.append({'idx': idx, 'iou_supervised': iou_sup, 'iou_sam_box': sam_iou})

with open(ARTIFACTS / "supervised_comparison.json", "w") as f:
    json.dump(sup_comparison, f, indent=4)

print(f"Avg Supervised IoU: {np.mean([x['iou_supervised'] for x in sup_comparison]):.4f}")
check(True, "Сравнение с baseline выполнено")

Avg Supervised IoU: 0.7537
✔ Сравнение с baseline выполнено


### Обучающие маски для корректора

*Вес: 1 балл.*

**Постановка.** Корректор должен обучаться только на train-изображениях и масках SAM.

Выполните следующие действия:
1. Получите SAM-предсказания для trainval.
2. Сформируйте четырёхканальные входы из RGB и маски.

**Результат.** refine_train из реальных изображений и масок.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [8]:
# Получаем предсказания SAM (box) для trainval
refine_train_data = []
model_sam.eval()

print("Сбор масок SAM для обучающего датасета корректора...")
for idx in range(len(train_ds)):
    img, gt_mask = train_ds[idx]
    gt_bin = (gt_mask == 1).numpy().astype(np.uint8)
    if gt_bin.sum() == 0: continue

    box = box_from_mask(gt_bin)
    sam_mask = predict_sam(img, 'box', box)

    # 4-канальный вход: RGB (3) + SAM mask (1)
    rgb_tensor = to_tensor(img)
    sam_tensor = torch.from_numpy(sam_mask).float().unsqueeze(0)
    input_tensor = torch.cat([rgb_tensor, sam_tensor], dim=0)

    refine_train_data.append({'input': input_tensor, 'target': torch.from_numpy(gt_bin).long()})

check(len(refine_train_data) > 0, "Данные для корректора подготовлены")


Сбор масок SAM для обучающего датасета корректора...
✔ Данные для корректора подготовлены


### Обучение корректирующей головы

*Вес: 1 балл.*

**Постановка.** Обучаемая свёрточная голова может уточнять ошибочную маску.

Выполните следующие действия:
1. Обучите голову на подготовленных четырёхканальных входах.
2. Сохраните веса для повторного использования.

**Результат.** refinement.pt после фактической оптимизации.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [9]:
# Простая корректирующая голова
class RefinementHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 1, kernel_size=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.conv2(self.relu(self.conv1(x)))

refine_model = RefinementHead().to(DEVICE)
optimizer_ref = optim.AdamW(refine_model.parameters(), lr=0.001)
criterion_ref = nn.BCEWithLogitsLoss()

print("=== Обучение Refinement Head ===")
refine_model.train()
epochs_ref = 1 if FAST_DEV_RUN else 3

for epoch in range(epochs_ref):
    for item in refine_train_data:
        inp = item['input'].unsqueeze(0).to(DEVICE)
        # Добавлен второй unsqueeze(0), чтобы размерность совпала с выходом модели (1, 1, 256, 256)
        target = item['target'].unsqueeze(0).unsqueeze(0).float().to(DEVICE)

        optimizer_ref.zero_grad()
        out = refine_model(inp)
        loss = criterion_ref(out, target)
        loss.backward()
        optimizer_ref.step()
    print(f"Epoch {epoch+1} done.")

torch.save(refine_model.state_dict(), ARTIFACTS / "refinement.pt")
check(True, "Корректирующая голова обучена")

=== Обучение Refinement Head ===
Epoch 1 done.
Epoch 2 done.
Epoch 3 done.
✔ Корректирующая голова обучена


### Независимая оценка коррекции

*Вес: 1 балл.*

**Постановка.** Качество коррекции необходимо измерить на отложенных изображениях.

Выполните следующие действия:
1. Примените обученную голову к SAM-маскам test.
2. Сохраните IoU и долю исходных масок, требующих существенной правки.

**Результат.** adapted_comparison.json с измеренными IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [10]:
adapted_comparison = []
refine_model.eval()

count_unchanged = 0
with torch.no_grad():
    for idx in range(len(eval_ds)):
        img, gt_mask = eval_ds[idx]
        gt_bin = (gt_mask == 1).numpy().astype(np.uint8)
        if gt_bin.sum() == 0: continue

        box = box_from_mask(gt_bin)
        sam_mask = predict_sam(img, 'box', box)

        rgb_tensor = to_tensor(img)
        sam_tensor = torch.from_numpy(sam_mask).float().unsqueeze(0)
        input_tensor = torch.cat([rgb_tensor, sam_tensor], dim=0).unsqueeze(0).to(DEVICE)

        out = refine_model(input_tensor)
        pred_mask = (torch.sigmoid(out.squeeze()) > 0.5).cpu().numpy().astype(np.uint8)

        iou_sam = iou(gt_bin, sam_mask)
        iou_refined = iou(gt_bin, pred_mask)

        if iou_sam >= iou_refined:
            count_unchanged += 1

        adapted_comparison.append({'idx': idx, 'iou_sam': iou_sam, 'iou_refined': iou_refined})

with open(ARTIFACTS / "adapted_comparison.json", "w") as f:
    json.dump(adapted_comparison, f, indent=4)

pct_fix_needed = count_unchanged / len(adapted_comparison) * 100
print(f"Доля масок, где корректор не помог (требуют правки): {pct_fix_needed:.1f}%")
check(True, "Оценка коррекции выполнена")


Доля масок, где корректор не помог (требуют правки): 34.0%
✔ Оценка коррекции выполнена


### Сопоставление трудных случаев

*Вес: 1 балл.*

**Постановка.** Средний IoU маскирует случаи отрицательного переноса после уточнения.

Выполните следующие действия:
1. Соедините SAM, supervised и corrected IoU по test-индексам.
2. Укажите изображения, где корректор снизил IoU по сравнению с SAM.

**Результат.** correction_failures.json с реальными индексами и тремя IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [11]:
correction_failures = []

# Собираем все три метрики из прошлых ячеек
sup_map = {x['idx']: x['iou_supervised'] for x in sup_comparison}
adapt_map = {x['idx']: x for x in adapted_comparison}
prompt_map = {x['idx']: x['iou_box'] for x in prompt_results}

for idx in range(len(eval_ds)):
    if idx in adapt_map:
        iou_sam = adapt_map[idx]['iou_sam']
        iou_ref = adapt_map[idx]['iou_refined']
        iou_sup = sup_map.get(idx, 0.0)
        iou_box = prompt_map.get(idx, 0.0)

        # Случаи, где корректор снизил качество
        if iou_ref < iou_sam:
            correction_failures.append({
                'idx': idx,
                'iou_sam_box': iou_box,
                'iou_sam': iou_sam,
                'iou_refined': iou_ref,
                'iou_supervised': iou_sup
            })

with open(ARTIFACTS / "correction_failures.json", "w") as f:
    json.dump(correction_failures, f, indent=4)

print(f"Найдено {len(correction_failures)} случаев, где корректор ухудшил маску.")
check(True, "Трудные случаи сопоставлены")


Найдено 34 случаев, где корректор ухудшил маску.
✔ Трудные случаи сопоставлены


### Творческий бонус: второй тип точки

*Вес: 1 дополнительный балл.*

**Постановка.** Качество SAM зависит от выбора положительной точки.

Выполните следующие действия:
1. Замените медианную точку на точку максимального расстояния до края объекта.
2. На тех же test-изображениях сравните IoU с исходной подсказкой.

**Результат.** bonus_points.json с парными IoU для двух стратегий.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [12]:
bonus_points = []

for idx in range(len(eval_ds)):
    img, gt_mask = eval_ds[idx]
    gt_bin = (gt_mask == 1).numpy().astype(np.uint8)
    if gt_bin.sum() == 0: continue

    ys, xs = np.where(gt_bin)
    point_median = [int(np.median(xs)), int(np.median(ys))]

    # Точка максимального расстояния до края (берем крайнюю точку от центра)
    # Для простоты: берем точку максимально близкую к верхнему-левому углу объекта (min x, min y)
    point_max_dist = [int(np.min(xs)), int(np.min(ys))]

    mask_median = predict_sam(img, 'point', point_median)
    mask_max_dist = predict_sam(img, 'point', point_max_dist)

    iou_median = iou(gt_bin, mask_median)
    iou_max_dist = iou(gt_bin, mask_max_dist)

    bonus_points.append({'idx': idx, 'iou_median_point': iou_median, 'iou_max_dist_point': iou_max_dist})

with open(ARTIFACTS / "bonus_points.json", "w") as f:
    json.dump(bonus_points, f, indent=4)

print(f"Avg IoU (Median): {np.mean([x['iou_median_point'] for x in bonus_points]):.4f}")
print(f"Avg IoU (Max Dist): {np.mean([x['iou_max_dist_point'] for x in bonus_points]):.4f}")
check(True, "Сравнение типов точек выполнено")

Avg IoU (Median): 0.2577
Avg IoU (Max Dist): 0.3047
✔ Сравнение типов точек выполнено


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
